#### MCP Client

Connects to the SSE server started in `005-mcp-server.ipynb` (running on `http://127.0.0.1:8000`).

In [22]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from pprint import pprint

Globals

In [23]:
model = f"bedrock:us.anthropic.claude-opus-4-6-v1"

Client

In [24]:
client = MultiServerMCPClient(
    {
        "local_server": {
            "transport": "sse",
            "url": "http://127.0.0.1:8000/sse",
        }
    }
)

Getting Prompts

In [25]:
prompt = await client.get_prompt("local_server", "prompt")
prompt

[HumanMessage(content='\n    You are a helpful assistant that answers user questions about the weather.\n\n    You can use the following tools/resources to answer user questions:\n    - get_forcast: Get weather forcast for a certain location\n    - get_alerts: Get weather alerts for a US state\n\n    If the user asks a question that is not related to the weather, you should say "I\'m sorry, I can only answer questions about LangChain, LangGraph and LangSmith."\n\n    You may try multiple tool and resource calls to answer the user\'s question.\n\n    You may also ask clarifying questions to the user to better understand their question.\n    ', additional_kwargs={}, response_metadata={})]

Getting Resources

In [26]:
resources = await client.get_resources()
resources

[]

Getting Tools

In [27]:
tools = await client.get_tools()
tools

[StructuredTool(name='get_alerts', description='Get weather alerts for a US state.\n\n    Args:\n        state: Two-letter US state code (e.g. CA, NY)\n    ', args_schema={'properties': {'state': {'title': 'State', 'type': 'string'}}, 'required': ['state'], 'title': 'get_alertsArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x115ab0c20>),
 StructuredTool(name='get_forecast', description='Get weather forecast for a location.\n\n    Args:\n        latitude: Latitude of the location\n        longitude: Longitude of the location\n    ', args_schema={'properties': {'latitude': {'title': 'Latitude', 'type': 'number'}, 'longitude': {'title': 'Longitude', 'type': 'number'}}, 'required': ['latitude', 'longitude'], 'title': 'get_forecastArguments', 'type': 'object'}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x104e02e80>)]

Agent

In [28]:
agent = create_agent(
    model,
    tools=tools,
    system_prompt=prompt[0].content
)

Prompt Protection

In [29]:
question = HumanMessage(content="What the weather in Japan?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What the weather in Japan?', additional_kwargs={}, response_metadata={}, id='030a8c2f-b6f0-4fb1-a5bf-12e46356864f'),
              AIMessage(content="I'm sorry, but I can only provide weather forecasts and alerts for locations within the **United States**. The tools I have access to use the US National Weather Service, which doesn't cover Japan.\n\nIf you'd like, I can help you with:\n- **Weather forecasts** for any US location (just give me a city or coordinates)\n- **Weather alerts** for any US state\n\nIs there a US location you'd like weather information for?", additional_kwargs={'usage': {'prompt_tokens': 830, 'completion_tokens': 100, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 930}, 'stop_reason': 'end_turn', 'model_id': 'us.anthropic.claude-opus-4-6-v1'}, response_metadata={'usage': {'prompt_tokens': 830, 'completion_tokens': 100, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 930

In [31]:
question = HumanMessage(content="What the weather in Seattle?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response["messages"][-1].content)

("Here's the weather forecast for **Seattle, WA**:\n"
 '\n'
 '🌧️ **Today:**\n'
 '- **Temperature:** 51°F\n'
 '- **Wind:** 7–24 mph S (gusts up to 45 mph)\n'
 '- **Forecast:** Rain and cloudy skies. 100% chance of precipitation with ¼ '
 'to ½ inch of rainfall possible.\n'
 '\n'
 '🌧️ **Tonight:**\n'
 '- **Temperature:** Low around 42°F (rising to ~44°F overnight)\n'
 '- **Wind:** 10–26 mph SW (gusts up to 45 mph)\n'
 '- **Forecast:** Rain showers and cloudy. 90% chance of precipitation.\n'
 '\n'
 '🌧️ **Thursday:**\n'
 '- **Temperature:** High near 45°F\n'
 '- **Wind:** 10 mph SW\n'
 '- **Forecast:** Rain and cloudy. 100% chance of precipitation.\n'
 '\n'
 '🌧️ **Thursday Night:**\n'
 '- **Temperature:** Low around 37°F\n'
 '- **Wind:** 2–9 mph SSW\n'
 '- **Forecast:** Rain. ½ to ¾ inch of new rainfall possible.\n'
 '\n'
 '🌧️ **Friday:**\n'
 '- **Temperature:** High near 43°F\n'
 '- **Wind:** 3 mph S\n'
 '- **Forecast:** Rain and cloudy. 90% chance of precipitation.\n'
 '\n'
 "Looks like 